# Implementação e Comparação do PSO
## Nome: Davi dos Santo Mattos
### DRE: 119133049

In [ ]:
import numpy as np
from random import random, seed

def rosenbrock(X):
  """
  Função de Rosenbrock

  X -> Vetor de entrada

  """
  X = np.array(X)
  if X.ndim > 1:
    return np.sum(100 * (X[:, 1:] - X[:, :-1]**2)**2 + (X[:, :-1] - 1)**2, axis=1)
  else:
    return np.sum(100 * (X[1:] - X[:-1]**2)**2 + (X[:-1] - 1)**2, axis=0)


def rastrigin(X):
    """
    Função de Rastrigin

    X -> Vetor de entrada ou população
    """
    X = np.array(X)

    if X.ndim > 1:
        n = X.shape[1]
        return 10 * n + np.sum(X**2 - 10 * np.cos(2 * np.pi * X), axis=1)
    else:
        n = len(X)
        return 10 * n + np.sum(X**2 - 10 * np.cos(2 * np.pi * X))

In [ ]:
def criar_populacao(dimensao, num_individuos, intervalo):
  """
  Função que cria uma população com N indivíduos.

  dimensao -> número de dimensões do problema
  n_individuos -> número de indivíduos na população
  intervalo -> intervalo de valores possíveis para cada dimensão

  """

  populacao = []
  for i in range(num_individuos):
    individuo = []
    for j in range(dimensao):
      individuo.append(np.random.uniform(intervalo[0], intervalo[1]))
    populacao.append(individuo)

  return np.array(populacao)

In [ ]:
def melhor_individuo(populacao, funcao_avaliacao):
  fitness = funcao_avaliacao(populacao)
  melhor_individuo = populacao[np.argmin(fitness)]
  return melhor_individuo

#### Segunda tentativa (limitando a velocidade)

In [ ]:
def pso(tam_populacao=100,problema=rosenbrock,intervalo=[-5, 10],dimensao=2,parada=10000, history=False):
    Vmax = (intervalo[1] - intervalo[0]) * 0.2  # limite de velocidade

    populacao = criar_populacao(dimensao, tam_populacao, intervalo)
    fitness   = problema(populacao)
    chamada_de_funcao = tam_populacao

    v = np.zeros((tam_populacao, dimensao))
    w = np.random.uniform(0.4, 0.9)
    c1 = 1.5
    c2 = 1.5

    #pbest
    pbest = populacao.copy()
    pbest_val = fitness.copy()

    #gbest
    idx_gbest = np.argmin(fitness)
    gbest = populacao[idx_gbest].copy()
    gbest_val = fitness[idx_gbest]

    historico = []


    while chamada_de_funcao < parada:
        for i in range(tam_populacao):

          # Atualizar velocidade
          r1 = np.random.rand(dimensao)
          r2 = np.random.rand(dimensao)
          v[i] = ( w  * v[i] + c1 * r1 * (pbest[i] - populacao[i]) + c2 * r2 * (gbest - populacao[i]) )

          v[i] = np.clip(v[i], -Vmax, Vmax)  # limita velocidade

          # Atualizar posição
          populacao[i] = np.clip(populacao[i] + v[i], intervalo[0], intervalo[1])

          # Reavaliar fitness
          fitness[i] = problema(populacao[i])
          w = 1 - chamada_de_funcao * (1 / parada)
          chamada_de_funcao += 1

          # Atualizar pBest
          if fitness[i] < pbest_val[i]:
              pbest_val[i] = fitness[i]
              pbest[i]     = populacao[i].copy()

              # Atualizar gBest
              if pbest_val[i] < gbest_val:
                  gbest_val = pbest_val[i]
                  gbest     = pbest[i].copy()
          historico.append(gbest_val)


    if history:
        return populacao, gbest, gbest_val, chamada_de_funcao, np.array(historico)
    else:
      print(f"Particula= [{gbest}] - f(x*) = {gbest_val}\nChamadas de Função = {chamada_de_funcao}")


In [ ]:
pso(parada=10000)

Particula= [[0.99998424 0.99996845]] - f(x*) = 2.4846539330300116e-10
Chamadas de Função = 10000


In [ ]:
pso(problema=rastrigin, intervalo=[-5.12, 5.12], dimensao=2, parada=10000)

Particula= [[ 3.65364810e-09 -1.17637092e-09]] - f(x*) = 0.0
Chamadas de Função = 10000


# EPSO


In [ ]:
def torneio(populacao, funcao_avaliacao, k, rng=None):
  """
  Retorna o melhor indivíduo da população e a nova população sem o melhor indivíduo

  populacao -> população de indivíduos
  funcao_avaliacao -> função de avaliação dos indivíduos
  k -> número de participantes no torneio

  """
  if rng is None:
    rng = np.random.default_rng()
  if populacao.ndim < k:
    individuos = populacao
  else:
    individuos = rng.choice(populacao, size=k, axis=0, replace=False)
  return individuos[np.argmin(individuos[:, -1])] , np.delete(individuos, np.argmin(individuos[:, -1]), axis=0)



In [ ]:
def mutacao(individuo, taxa_mutacao=0.05, sigma=0.1, rng=None):
  """
  Realizar a mutação em um indivíduo utilizando a distribuição normal(0,sigma)

  individuo -> indivíduo a ser mutado
  taxa_mutacao -> taxa de mutação
  sigma -> desvio padrão da distribuição normal

  """
  if rng is None:
    rng = np.random.default_rng()

  novo_individuo = individuo.copy()

  for i in range(len(individuo)):
    if rng.random() < taxa_mutacao:
      novo_individuo[i] += rng.normal(0, sigma)

  return novo_individuo

In [ ]:
def epso(tam_populacao=100,problema=rosenbrock,intervalo=[-5,10],dimensao=2,parada=10000,pc=0.5,history=False):
  tau = 1.0 / np.sqrt(2.0 * dimensao)
  Vmax = (intervalo[1] - intervalo[0]) * 0.2

  populacao = criar_populacao(dimensao, tam_populacao, intervalo)
  fitness = problema(populacao)
  chamada_de_funcao = tam_populacao

  v = np.zeros((tam_populacao, dimensao))

  wi = np.random.uniform(0.4, 0.9, size=tam_populacao)
  em = np.full(tam_populacao, 1.5)
  wc = np.full(tam_populacao, 1.5)

  pbest = populacao.copy()
  pbest_val = fitness.copy()

  idx_gbest = np.argmin(fitness)
  gbest = populacao[idx_gbest].copy()
  gbest_val = fitness[idx_gbest]

  historico = []

  while chamada_de_funcao < parada:
    for i in range(tam_populacao):

      # Mutação
      wi_filho = wi[i] * np.exp(-tau * chamada_de_funcao)
      em_filho = em[i] * np.exp(-tau * chamada_de_funcao)
      wc_filho = wc[i] * np.exp(-tau * chamada_de_funcao)

      wi_filho = np.clip(wi_filho, 0.4, 0.9)
      em_filho = np.clip(em_filho, 0.5, 2.5)
      wc_filho = np.clip(wc_filho, 0.5, 2.5)

      matrizC = (np.random.rand(dimensao) < pc).astype(float)

      # Velocidades
      r1 = np.random.rand(dimensao)
      r2 = np.random.rand(dimensao)

      v_filho = (
          wi_filho * v[i]
          + em_filho * (pbest[i] - populacao[i])
          + wc_filho * matrizC * (gbest - populacao[i])
      )
      v_filho = np.clip(v_filho, -Vmax, Vmax)

      # Atualizar posição
      candidato = np.clip(populacao[i] + v_filho, intervalo[0], intervalo[1])

      # Avaliar Candidato
      fit_candidado = problema(candidato)
      chamada_de_funcao += 1

      # Seleção

      if fit_candidado < fitness[i]:
        populacao[i] = candidato
        fitness[i] = fit_candidado
        v[i] = v_filho

        # Atualiza parâmetros
        wi[i] = wi_filho
        em[i] = em_filho
        wc[i] = wc_filho

        # Atualiza pbest
        if fitness[i] < pbest_val[i]:
          pbest_val[i] = fitness[i]
          pbest[i] = populacao[i].copy()

          # Atualiza gbest
          if fitness[i] < gbest_val:
            gbest_val = fitness[i]
            gbest = populacao[i].copy()

    historico.append(gbest_val)

  if history:
    return populacao, gbest, gbest_val, chamada_de_funcao, np.array(historico)
  else:
    print(f"Indivíduo = {gbest}")
    print(f"f(x*)     = {gbest_val}")
    print(f"Chamadas  = {chamada_de_funcao}")

In [ ]:
epso(tam_populacao=100, problema=rosenbrock, intervalo=[-5,10], parada=10000)

Indivíduo = [0.99989023 0.99977306]
f(x*)     = 1.7553314992550977e-08
Chamadas  = 10000


In [ ]:
epso(tam_populacao=100, problema=rastrigin, intervalo=[-5.12,5.12], dimensao=2, parada=10000)

Indivíduo = [1.13259713e-04 7.00644434e-05]
f(x*)     = 3.5188384401863004e-06
Chamadas  = 10000
